# Grasp adaptation — compliance sensing + gradient-descent force control

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as _mpatch
import os, sys
from pathlib import Path

sys.path.insert(0, os.path.join('../'))
from plot_config import draw_radar
from hand_config import (K_TIP_GENTLE, K_TIP_PROBE, K_TIP_HOLD,
                         F_GAIN, GD_LR, F_CONVERGE_THR)
from ModelIDHand.hand_viz import plot_hand, _COL

OUTPUT_DIR  = os.path.join('outputs', 'grasp_adaptation')
os.makedirs(OUTPUT_DIR, exist_ok=True)

FINGERTIPS   = ['thumb', 'index', 'middle', 'ring', 'pinky']
OBJECTS      = ['hard_obj', 'soft_obj']
MODES        = ['stiffness', 'ref']
OBJ_LABELS   = {'hard_obj': 'Hard', 'soft_obj': 'Soft'}
MODE_LABELS  = {'stiffness': 'Stiffness descent', 'ref': 'Ref descent'}
OBJ_COLOR    = {'hard_obj': 'C0', 'soft_obj': 'C1'}
MODE_LS      = {'stiffness': '-', 'ref': '--'}

_Q_COLS    = [f'q_motor_{i}_rad' for i in range(13)]
_STYLE     = dict(linestyle='-', linewidth=4.5, marker='o', markersize=6.0)
_ELEV, _AZIM = 22, 112


def load(obj, mode):
    folder = Path(OUTPUT_DIR)
    if not folder.exists():
        return None
    files = sorted(folder.glob(f'grasp_{obj}_{mode}.csv'))
    return pd.read_csv(files[-1]) if files else None


def _steady(df, phase, frac=0.2):
    rows = df[df['phase'] == phase]
    return rows.tail(max(1, int(len(rows) * frac))) if not rows.empty else rows


def _polish_ax(ax, title):
    ax.set_xticks([]); ax.set_yticks([]); ax.set_zticks([])
    ax.set_xlabel(''); ax.set_ylabel(''); ax.set_zlabel('')
    for pane in (ax.xaxis.pane, ax.yaxis.pane, ax.zaxis.pane):
        pane.fill = False
        pane.set_edgecolor('#d8d8d8')
    for axis in (ax.xaxis, ax.yaxis, ax.zaxis):
        axis._axinfo['grid']['color']     = (0.88, 0.88, 0.88, 0.6)
        axis._axinfo['grid']['linewidth'] = 0.4
    ax.view_init(elev=_ELEV, azim=_AZIM)
    ax.set_title(title, fontsize=plt.rcParams['axes.titlesize'],
                 fontweight='semibold', pad=14, color='#1a1a1a')


def _sync_axes(axes):
    lims = np.array([[a.get_xlim3d(), a.get_ylim3d(), a.get_zlim3d()] for a in axes])
    xl = [lims[:,0,0].min(), lims[:,0,1].max()]
    yl = [lims[:,1,0].min(), lims[:,1,1].max()]
    zl = [lims[:,2,0].min(), lims[:,2,1].max()]
    span = max(xl[1]-xl[0], yl[1]-yl[0], zl[1]-zl[0])
    cx, cy, cz = 0.5*sum(xl), 0.5*sum(yl), 0.5*sum(zl)
    for a in axes:
        a.set_xlim3d(cx-span/2, cx+span/2)
        a.set_ylim3d(cy-span/2, cy+span/2)
        a.set_zlim3d(cz-span/2, cz+span/2)


data = {obj: {mode: load(obj, mode) for mode in MODES} for obj in OBJECTS}
print('Loaded:')
for obj in OBJECTS:
    for mode in MODES:
        df = data[obj][mode]
        print(f'  {obj}/{mode}: {len(df)} rows' if df is not None else f'  {obj}/{mode}: no data')

## Compliance classification

$C_O = \|\Delta x_{\rm thumb}\| / \|\Delta F_{\rm thumb}\|$ (sense $\to$ probe finite difference).
Lower $C_O$ = stiffer object. Annotated with the resulting target force $f_{\rm des} = F_{\rm GAIN} / C_O$.

In [ ]:
fig, ax = plt.subplots()

x     = np.arange(len(OBJECTS))
width = 0.35

for mi, mode in enumerate(MODES):
    C_Os, f_des_vals = [], []
    for obj in OBJECTS:
        df = data[obj][mode]
        if df is None:
            C_Os.append(np.nan); f_des_vals.append(np.nan); continue
        probe = df[df['phase'] == 'probe']
        C_O   = float(probe['C_O_m_per_N'].median()) if not probe.empty else np.nan
        C_Os.append(C_O * 1e3)
        f_des_vals.append(F_GAIN / C_O if C_O > 1e-12 else np.nan)

    bars = ax.bar(x + (mi - 0.5) * width, C_Os, width,
                  color=[OBJ_COLOR[o] for o in OBJECTS],
                  alpha=(1.0 if mode == 'stiffness' else 0.5),
                  label=MODE_LABELS[mode])
    for bar, C_O_val, f_des_val in zip(bars, C_Os, f_des_vals):
        if np.isnan(C_O_val):
            continue
        ax.text(bar.get_x() + bar.get_width() / 2,
                bar.get_height() + 0.3,
                f'${C_O_val:.1f}$ mm/N\n$\\to f_{{\\rm des}}={f_des_val:.2f}$ N',
                ha='center', va='bottom', fontsize=14)

ax.set_xticks(x)
ax.set_xticklabels([OBJ_LABELS[o] for o in OBJECTS])
ax.set_ylabel(r'$C_O^{\rm thumb}$ [mm/N]')
ax.set_ylim(0, None)
ax.legend()
fig.tight_layout()
fig.savefig(os.path.join(OUTPUT_DIR, 'compliance_classification.pdf'))
plt.show()

## Radar: fingertip force distribution — GD start vs end

Four panels (2 objects $\times$ 2 modes). Each radar shows $|F_{\rm tip}|$ per finger
at the **start** (first 20 % of `adapt_gd`) and **end** (last 20 %) of the gradient-descent phase.
Spokes are normalised to the per-finger maximum across both traces.

In [ ]:
fig, axes = plt.subplots(2, 2, subplot_kw=dict(projection='polar'),
                         figsize=(14, 12))

for i, obj in enumerate(OBJECTS):
    for j, mode in enumerate(MODES):
        ax  = axes[i, j]
        df  = data[obj][mode]
        if df is None:
            ax.set_title(f'{OBJ_LABELS[obj]} / {MODE_LABELS[mode]}\n(no data)')
            continue

        gd = df[df['phase'] == 'adapt_gd']
        if gd.empty:
            ax.set_title(f'{OBJ_LABELS[obj]} / {MODE_LABELS[mode]}\n(no adapt_gd)')
            continue

        n    = max(1, len(gd) // 5)
        snap = {'start': gd.head(n), 'end': gd.tail(n)}

        traces = []
        for snap_label, rows in snap.items():
            forces = [float(rows[f'force_1st_{f}_mag_N'].mean()) for f in FINGERTIPS]
            traces.append({
                'label':  snap_label.capitalize(),
                'values': forces,
                'color':  OBJ_COLOR[obj],
                'lw':     1.8 if snap_label == 'start' else 2.8,
                'alpha':  0.12 if snap_label == 'start' else 0.22,
            })
        # ghost (start) has lighter style
        traces[0]['lw']    = 1.8
        traces[0]['alpha'] = 0.10

        draw_radar([f.capitalize() for f in FINGERTIPS], traces, ax=ax)

        f_des_mag = float(gd['f_des_mag_N'].iloc[0])
        ax.set_title(
            f'{OBJ_LABELS[obj]} — {MODE_LABELS[mode]}\n'
            f'$f_{{\\rm des}}={f_des_mag:.2f}$ N',
            fontsize=plt.rcParams['axes.titlesize'], fontweight='semibold',
            pad=18, color='#1a1a1a')

        handles = [
            _mpatch.Patch(color=OBJ_COLOR[obj], alpha=0.4, label='Start'),
            _mpatch.Patch(color=OBJ_COLOR[obj], alpha=0.9, label='End'),
        ]
        ax.legend(handles=handles, loc='upper center',
                  bbox_to_anchor=(0.5, 1.18), ncol=2, frameon=False)

fig.tight_layout()
fig.savefig(os.path.join(OUTPUT_DIR, 'radar_force_gd.pdf'))
plt.show()

## Thumb force evolution during GD

$|F_{\rm thumb}|$ over time in the `adapt_gd` phase. Dashed line = $f_{\rm des}$ target.
One panel per object, both modes overlaid.

In [ ]:
fig, axes = plt.subplots(1, 2, sharey=True)

for ax, obj in zip(axes, OBJECTS):
    f_des_drawn = False
    for mode in MODES:
        df = data[obj][mode]
        if df is None:
            continue
        gd = df[df['phase'] == 'adapt_gd']
        if gd.empty:
            continue
        t0    = gd['time_s'].iloc[0]
        f_mag = gd['f_meas_gd_mag_N'].to_numpy()
        ax.plot(gd['time_s'] - t0, f_mag,
                color=OBJ_COLOR[obj], ls=MODE_LS[mode], lw=2.5,
                label=MODE_LABELS[mode])
        if not f_des_drawn:
            f_des = float(gd['f_des_mag_N'].iloc[0])
            ax.axhline(f_des, color='0.4', lw=1.5, ls=':',
                       label=r'$f_{\rm des}=' + f'{f_des:.2f}$ N')
            f_des_drawn = True

    ax.set_title(OBJ_LABELS[obj], fontweight='semibold')
    ax.set_xlabel('Time in GD phase [s]')
    ax.legend()

axes[0].set_ylabel(r'$|F_{\rm thumb}|$ [N]')
fig.tight_layout()
fig.savefig(os.path.join(OUTPUT_DIR, 'thumb_force_gd.pdf'))
plt.show()

## Thumb task stiffness evolution during GD

**Stiffness mode**: mean diagonal of $K_{\rm thumb}^{\rm task}$ [N/m] — the parameter being updated.  
**Ref mode**: $K_{\rm thumb}^{\rm task}$ is held fixed at $K_{\rm probe}$; the virtual equilibrium $d_{\rm ref}^{\rm thumb}$ moves instead (displacement shown).

In [ ]:
fig, axes = plt.subplots(1, 2)
ax_stiff, ax_ref = axes

for obj in OBJECTS:
    # ── stiffness mode ────────────────────────────────────────────────────
    df = data[obj]['stiffness']
    if df is not None:
        gd = df[df['phase'] == 'adapt_gd']
        if not gd.empty:
            t0     = gd['time_s'].iloc[0]
            K_mean = gd[['K_thumb_task_00_Npm',
                          'K_thumb_task_11_Npm',
                          'K_thumb_task_22_Npm']].mean(axis=1)
            ax_stiff.plot(gd['time_s'] - t0, K_mean,
                          color=OBJ_COLOR[obj], lw=2.5,
                          label=OBJ_LABELS[obj])

    # ── ref mode ──────────────────────────────────────────────────────────
    df = data[obj]['ref']
    if df is not None:
        gd = df[df['phase'] == 'adapt_gd']
        if not gd.empty:
            t0    = gd['time_s'].iloc[0]
            d0    = gd[['d_ref_thumb_x_m','d_ref_thumb_y_m','d_ref_thumb_z_m']].iloc[0].to_numpy()
            disp  = np.linalg.norm(
                gd[['d_ref_thumb_x_m','d_ref_thumb_y_m','d_ref_thumb_z_m']].to_numpy() - d0,
                axis=1) * 1e3
            ax_ref.plot(gd['time_s'] - t0, disp,
                        color=OBJ_COLOR[obj], lw=2.5,
                        label=OBJ_LABELS[obj])

ax_stiff.axhline(K_TIP_PROBE, color='0.5', lw=1.5, ls='--',
                 label=r'$K_{\rm probe}$')
ax_stiff.set_ylabel(r'$\bar{K}_{\rm thumb}^{\rm task}$ [N/m]')
ax_stiff.set_xlabel('Time in GD phase [s]')
ax_stiff.set_title('Stiffness descent', fontweight='semibold')
ax_stiff.legend()

ax_ref.set_ylabel(r'$\|\Delta d_{\rm ref}^{\rm thumb}\|$ [mm]')
ax_ref.set_xlabel('Time in GD phase [s]')
ax_ref.set_title('Ref descent', fontweight='semibold')
ax_ref.legend()

fig.tight_layout()
fig.savefig(os.path.join(OUTPUT_DIR, 'thumb_stiffness_gd.pdf'))
plt.show()

## Hand pose during lift

Motor configuration $q$ at the midpoint of the `lift` phase, one panel per object $\times$ mode.

In [ ]:
panels = [(obj, mode)
          for obj in OBJECTS for mode in MODES
          if data[obj][mode] is not None
          and not data[obj][mode][data[obj][mode]['phase'] == 'lift'].empty]

if panels:
    fig = plt.figure(figsize=(6 * len(panels), 6))
    fig.patch.set_facecolor('white')
    axes3d = []
    for i, (obj, mode) in enumerate(panels):
        df   = data[obj][mode]
        lift = df[df['phase'] == 'lift']
        q    = lift[_Q_COLS].iloc[len(lift) // 2].to_numpy(dtype=np.float64)
        ax   = fig.add_subplot(1, len(panels), i + 1, projection='3d')
        plot_hand(q, ax=ax, style=_STYLE)
        leg = ax.get_legend()
        if leg: leg.remove()
        f_des = float(lift['f_des_mag_N'].iloc[0])
        _polish_ax(ax, f'{OBJ_LABELS[obj]} — {MODE_LABELS[mode]}\n$f_{{\\rm des}}={f_des:.2f}$ N')
        axes3d.append(ax)

    _sync_axes(axes3d)
    _LEG = [plt.Line2D([0],[0], color=_COL[f], lw=5, label=f.capitalize())
            for f in FINGERTIPS]
    fig.legend(handles=_LEG, loc='lower center', ncol=5,
               fontsize=plt.rcParams['legend.fontsize'], frameon=False,
               bbox_to_anchor=(0.5, -0.02), handlelength=3.0, columnspacing=1.4)
    fig.tight_layout()
    fig.savefig(os.path.join(OUTPUT_DIR, 'lift_hand_pose.pdf'), dpi=300)
    plt.show()
else:
    print('No lift-phase data found.')